Importing Libraries

In [3]:
import os
import zipfile
import random
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report

random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

zip_path = "D:/Deepfake-detection/datasets/asvpoof-2019-dataset.zip"
asv_base = "D:/Deepfake-detection/datasets/asvspoof_extracted"
spec_asv_base = "D:/Deepfake-detection/datasets/spectrograms_asvspoof"

IMG_SIZE = 224
BATCH_SIZE = 16

asv_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def parse_protocol(zip_ref, protocol_path):
    with zip_ref.open(protocol_path) as f:
        lines = f.read().decode('utf-8').splitlines()
    mapping = {}
    for line in lines:
        parts = line.split()
        filename = parts[1]
        label = parts[-1]
        mapping[filename] = label
    return mapping

def get_balanced_filenames(labels_dict, samples_per_class=2500):
    bonafide_files = [f for f, l in labels_dict.items() if l == 'bonafide']
    spoof_files = [f for f, l in labels_dict.items() if l == 'spoof']
    n_bonafide = min(samples_per_class, len(bonafide_files))
    n_spoof = min(samples_per_class, len(spoof_files))
    return random.sample(bonafide_files, n_bonafide), random.sample(spoof_files, n_spoof)

def build_filename_lookup(entries_list):
    lookup = {}
    for entry in entries_list:
        stem = os.path.basename(entry).replace(".flac", "")
        lookup[stem] = entry
    return lookup

def extract_selected(zip_ref, filenames, lookup, dest_folder):
    os.makedirs(dest_folder, exist_ok=True)
    extracted, missing = 0, 0
    for fname in tqdm(filenames, desc=f"Extracting to {os.path.basename(dest_folder)}"):
        if fname not in lookup:
            missing += 1
            continue
        entry = lookup[fname]
        out_path = os.path.join(dest_folder, fname + ".flac")
        with zip_ref.open(entry) as source, open(out_path, 'wb') as target:
            target.write(source.read())
        extracted += 1
    return extracted, missing

def audio_to_spectrogram_image_v3(audio_path, output_path, sr=16000, n_mels=128):
    try:
        y, _ = librosa.load(audio_path, sr=sr)
    except Exception as e:
        print(f"Failed to load {audio_path}: {e}")
        return False
    if len(y) == 0:
        return False
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    fig, ax = plt.subplots(figsize=(2.24, 2.24), dpi=100)
    ax.axis('off')
    librosa.display.specshow(mel_spec_db, sr=sr, ax=ax, cmap='magma')
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    fig.savefig(output_path, dpi=100)
    plt.close(fig)
    return True

def convert_folder_v3(source_folder, dest_folder):
    os.makedirs(dest_folder, exist_ok=True)
    files = [f for f in os.listdir(source_folder) if f.endswith('.flac')]
    success, fail = 0, 0
    for f in tqdm(files, desc=f"Converting {os.path.basename(source_folder)}"):
        out_path = os.path.join(dest_folder, f.replace('.flac', '.png'))
        if audio_to_spectrogram_image_v3(os.path.join(source_folder, f), out_path):
            success += 1
        else:
            fail += 1
    return success, fail

print("All functions and paths ready.")

Using device: cuda
All functions and paths ready.


Step 2 - Reload the trained model

In [4]:
asv_model = models.efficientnet_b0(weights=None)
num_features = asv_model.classifier[1].in_features
asv_model.classifier[1] = nn.Linear(num_features, 2)
asv_model.load_state_dict(torch.load("D:/Deepfake-detection/saved_models/asvspoof_efficientnet_best.pth", map_location=device))
asv_model = asv_model.to(device)
asv_model.eval()

asv_train_data = datasets.ImageFolder(f"{spec_asv_base}/train", transform=asv_transform)
print("Classes:", asv_train_data.classes)
print("Model reloaded.")

Classes: ['bonafide', 'spoof']
Model reloaded.


Step 3 - Get the LA audio entries list again (needed for eval extraction)

In [5]:
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    names = zip_ref.namelist()

la_audio_entries = [n for n in names if n.endswith('.flac') and n.startswith("LA/LA/")]
print(f"LA audio entries: {len(la_audio_entries)}")

LA audio entries: 122299


Redefine the paths (since the notebook is fresh, these variables need to exist again)

In [13]:
asv_base = "D:/Deepfake-detection/datasets/asvspoof_extracted"
spec_asv_base = "D:/Deepfake-detection/datasets/spectrograms_asvspoof"

inspect the structure of dataset

What we're checking: ASVspoof 2019 has two main subsets — LA (Logical Access), focused on TTS/voice-conversion synthetic speech (this is the one relevant to deepfake detection), and PA (Physical Access), focused on replay attacks (recording a real voice and playing it back — a different threat model, less relevant here). We want to confirm the folder structure so we can filter to just the LA subset and its train/dev/eval splits, ignoring PA.

In [1]:
import zipfile

zip_path = "D:/Deepfake-detection/datasets/asvpoof-2019-dataset.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    names = zip_ref.namelist()
    print(f"Total entries in zip: {len(names)}")
    print(names[:15])

Total entries in zip: 363397
['LA/LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.dev.female.trl.txt', 'LA/LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.dev.female.trn.txt', 'LA/LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.dev.gi.trl.txt', 'LA/LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.dev.male.trl.txt', 'LA/LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.dev.male.trn.txt', 'LA/LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.eval.female.trl.txt', 'LA/LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.eval.female.trn.txt', 'LA/LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.eval.gi.trl.txt', 'LA/LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.eval.male.trl.txt', 'LA/LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.eval.male.trn.txt', 'LA/LA/ASVspoof2019_LA_asv_scores/ASVspoof2019.LA.asv.dev.gi.trl.scores.txt', 'LA/LA/ASVspoof2019_LA_asv_scores/ASVspoof2019.LA.asv.eval.gi.trl.scores.txt', 'LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof

Find the actual audio file entries and the protocol/label files we need

What this does: ASVspoof stores audio as .flac files (a lossless format, notably NOT the mixed-encoding mess we had with FoR — a good early sign), and separately provides protocol files (plain text) that map each audio filename to its label (bonafide = real, spoof = fake) and speaker metadata. This separation of "audio" from "labels" via an official protocol file — rather than just sorting into real//fake/ folders — is actually part of what makes ASVspoof more rigorous: the labeling is explicit and auditable, not just implied by folder placement.

In [2]:
audio_entries = [n for n in names if n.endswith('.flac')]
protocol_entries = [n for n in names if 'cm_protocols' in n]

print(f"Total audio (.flac) files: {len(audio_entries)}")
print(audio_entries[:5])
print()
print("Protocol files (tell us real vs fake labels):")
for p in protocol_entries:
    print(p)

Total audio (.flac) files: 363355
['LA/LA/ASVspoof2019_LA_dev/flac/LA_D_1000265.flac', 'LA/LA/ASVspoof2019_LA_dev/flac/LA_D_1000752.flac', 'LA/LA/ASVspoof2019_LA_dev/flac/LA_D_1001095.flac', 'LA/LA/ASVspoof2019_LA_dev/flac/LA_D_1002130.flac', 'LA/LA/ASVspoof2019_LA_dev/flac/LA_D_1002200.flac']

Protocol files (tell us real vs fake labels):
LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt
LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt
LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt
PA/PA/ASVspoof2019_PA_cm_protocols/ASVspoof2019.PA.cm.dev.trl.txt
PA/PA/ASVspoof2019_PA_cm_protocols/ASVspoof2019.PA.cm.eval.trl.txt
PA/PA/ASVspoof2019_PA_cm_protocols/ASVspoof2019.PA.cm.train.trn.txt


Filter to LA subset only, and peek at the train protocol file

What this does: narrows down to just the LA subset's audio (ignoring PA), and reads the training protocol file's raw text content directly from inside the zip — no extraction needed just to peek at it, same "look before you extract" approach we've used throughout.

In [3]:
la_audio_entries = [n for n in audio_entries if n.startswith("LA/LA/")]
print(f"LA subset audio files: {len(la_audio_entries)}")

# Read the training protocol file directly from the zip (small text file, no need to extract yet)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    with zip_ref.open("LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt") as f:
        lines = f.read().decode('utf-8').splitlines()

print(f"Total lines in train protocol: {len(lines)}")
print("First 5 lines:")
for line in lines[:5]:
    print(line)

LA subset audio files: 122299
Total lines in train protocol: 25380
First 5 lines:
LA_0079 LA_T_1138215 - - bonafide
LA_0079 LA_T_1271820 - - bonafide
LA_0079 LA_T_1272637 - - bonafide
LA_0079 LA_T_1276960 - - bonafide
LA_0079 LA_T_1341447 - - bonafide


parse all three protocol files (train, dev, eval) into clean filename→label mappings

What this does: builds three dictionaries (train_labels, dev_labels, eval_labels) mapping each audio filename directly to its official bonafide/spoof label, straight from ASVspoof's own protocol — not inferred from folder names or any file metadata that could carry hidden confounds

Note on class balance: ASVspoof is known to be imbalanced (many more spoof samples than bonafide, since multiple different TTS/voice-conversion systems generated fakes against a smaller pool of real recordings) — the Counter output will show us exactly how imbalanced, which we'll need to account for during training (e.g., using a sampling strategy or class weighting) rather than assuming a clean 50/50 split like FoR had.

In [4]:
def parse_protocol(zip_ref, protocol_path):
    with zip_ref.open(protocol_path) as f:
        lines = f.read().decode('utf-8').splitlines()
    
    mapping = {}
    for line in lines:
        parts = line.split()
        filename = parts[1]  # e.g., LA_T_1138215
        label = parts[-1]    # bonafide or spoof
        mapping[filename] = label
    return mapping

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    train_labels = parse_protocol(zip_ref, "LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt")
    dev_labels = parse_protocol(zip_ref, "LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt")
    eval_labels = parse_protocol(zip_ref, "LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt")

print("Train labels:", len(train_labels))
print("Dev labels:", len(dev_labels))
print("Eval labels:", len(eval_labels))

# Check class balance
from collections import Counter
print("Train balance:", Counter(train_labels.values()))
print("Dev balance:", Counter(dev_labels.values()))
print("Eval balance:", Counter(eval_labels.values()))

Train labels: 25380
Dev labels: 24844
Eval labels: 71237
Train balance: Counter({'spoof': 22800, 'bonafide': 2580})
Dev balance: Counter({'spoof': 22296, 'bonafide': 2548})
Eval balance: Counter({'spoof': 63882, 'bonafide': 7355})


Build a balanced sample from train and dev

Why 2,500 per class: train only has 2,580 bonafide total, so this uses nearly all of them while downsampling spoof to match — giving us a clean, balanced ~5,000 training samples, avoiding the imbalance shortcut entirely.

In [5]:
import random
random.seed(42)

def get_balanced_filenames(labels_dict, samples_per_class=2500):
    bonafide_files = [f for f, l in labels_dict.items() if l == 'bonafide']
    spoof_files = [f for f, l in labels_dict.items() if l == 'spoof']
    
    n_bonafide = min(samples_per_class, len(bonafide_files))
    n_spoof = min(samples_per_class, len(spoof_files))
    
    sampled_bonafide = random.sample(bonafide_files, n_bonafide)
    sampled_spoof = random.sample(spoof_files, n_spoof)
    
    return sampled_bonafide, sampled_spoof

train_bonafide, train_spoof = get_balanced_filenames(train_labels, samples_per_class=2500)
dev_bonafide, dev_spoof = get_balanced_filenames(dev_labels, samples_per_class=2500)

print(f"Train: {len(train_bonafide)} bonafide, {len(train_spoof)} spoof")
print(f"Dev: {len(dev_bonafide)} bonafide, {len(dev_spoof)} spoof")

Train: 2500 bonafide, 2500 spoof
Dev: 2500 bonafide, 2500 spoof


Extract the selected audio files, guided by filename mapping (not folder-guessing)

Why we split by folder first: searching through all 122,299 filenames for every one of our ~10,000 selected files would be slow (nested loop). Narrowing to just the relevant folder's entries first makes the lookup much faster.

In [6]:
import os
from tqdm import tqdm

def find_full_path(filename, entries_list):
    """The protocol gives us short filenames like LA_T_1138215 — find the matching full zip path."""
    for entry in entries_list:
        if filename in entry:
            return entry
    return None

# Split la_audio_entries by which folder they're in, to speed up lookups
train_entries = [e for e in la_audio_entries if "ASVspoof2019_LA_train" in e]
dev_entries = [e for e in la_audio_entries if "ASVspoof2019_LA_dev" in e]

print(f"Train folder entries: {len(train_entries)}")
print(f"Dev folder entries: {len(dev_entries)}")

Train folder entries: 25380
Dev folder entries: 24986


Build a fast filename→path dictionary, then extract

What this does: builds a dictionary where each key is the short filename (e.g., LA_T_1138215) and the value is the full path inside the zip — turning what would be a slow linear search into an instant dictionary lookup, since we need to resolve ~10,000 filenames against tens of thousands of entries.

In [7]:
def build_filename_lookup(entries_list):
    lookup = {}
    for entry in entries_list:
        # e.g. "LA/LA/ASVspoof2019_LA_train/flac/LA_T_1138215.flac" -> key "LA_T_1138215"
        base = os.path.basename(entry)
        stem = base.replace(".flac", "")
        lookup[stem] = entry
    return lookup

train_lookup = build_filename_lookup(train_entries)
dev_lookup = build_filename_lookup(dev_entries)

print(f"Train lookup size: {len(train_lookup)}")
print(f"Dev lookup size: {len(dev_lookup)}")

Train lookup size: 25380
Dev lookup size: 24986


Extract the selected files into organized folders

What this does: extracts exactly the 2,500+2,500+2,500+2,500 = 10,000 files we selected, organized into clean train/bonafide, train/spoof, validation/bonafide, validation/spoof folders — driven entirely by the authoritative protocol-file labels, not folder-name guessing.

In [8]:
def extract_selected(zip_ref, filenames, lookup, dest_folder):
    os.makedirs(dest_folder, exist_ok=True)
    extracted = 0
    missing = 0
    for fname in tqdm(filenames, desc=f"Extracting to {os.path.basename(dest_folder)}"):
        if fname not in lookup:
            missing += 1
            continue
        entry = lookup[fname]
        out_path = os.path.join(dest_folder, fname + ".flac")
        with zip_ref.open(entry) as source, open(out_path, 'wb') as target:
            target.write(source.read())
        extracted += 1
    return extracted, missing

asv_base = "D:/Deepfake-detection/datasets/asvspoof_extracted"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    e1, m1 = extract_selected(zip_ref, train_bonafide, train_lookup, f"{asv_base}/train/bonafide")
    e2, m2 = extract_selected(zip_ref, train_spoof, train_lookup, f"{asv_base}/train/spoof")
    e3, m3 = extract_selected(zip_ref, dev_bonafide, dev_lookup, f"{asv_base}/validation/bonafide")
    e4, m4 = extract_selected(zip_ref, dev_spoof, dev_lookup, f"{asv_base}/validation/spoof")

print(f"train/bonafide: {e1} extracted, {m1} missing")
print(f"train/spoof: {e2} extracted, {m2} missing")
print(f"validation/bonafide: {e3} extracted, {m3} missing")
print(f"validation/spoof: {e4} extracted, {m4} missing")

Extracting to spoof: 100%|██████████| 2500/2500 [00:02<00:00, 1145.73it/s]

train/bonafide: 2500 extracted, 0 missing
train/spoof: 2500 extracted, 0 missing
validation/bonafide: 2500 extracted, 0 missing
validation/spoof: 2500 extracted, 0 missing


Convert to spectrograms (reusing your existing function, adapted for .flac)

A spectrogram is a visual picture of sound that shows frequencies changing over time. In machine learning, it turns raw audio into a 2D image, letting powerful computer vision models like Convolutional Neural Networks (CNNs) process speech, music, and noise

How a Spectrogram Works
Splitting Time: Audio is cut into very short, overlapping slices.

Finding Frequencies: A math tool called the Fourier Transform finds the power of different pitches in each slice.

The Graph: Time goes on the horizontal x-axis, frequency goes on the vertical y-axis, and color brightness shows loudness (z-axis).

In [14]:
def audio_to_spectrogram_image_v3(audio_path, output_path, sr=16000, n_mels=128):
    try:
        y, _ = librosa.load(audio_path, sr=sr)
    except Exception:
        return False
    
    if len(y) == 0:
        return False
    
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    fig, ax = plt.subplots(figsize=(2.24, 2.24), dpi=100)
    ax.axis('off')
    librosa.display.specshow(mel_spec_db, sr=sr, ax=ax, cmap='magma')
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    fig.savefig(output_path, dpi=100)
    plt.close(fig)
    return True

Run conversion across all 4 folders

In [15]:
def convert_folder_v3(source_folder, dest_folder):
    os.makedirs(dest_folder, exist_ok=True)
    files = [f for f in os.listdir(source_folder) if f.endswith('.flac')]
    success, fail = 0, 0
    for f in tqdm(files, desc=f"Converting {os.path.basename(source_folder)}"):
        out_path = os.path.join(dest_folder, f.replace('.flac', '.png'))
        if audio_to_spectrogram_image_v3(os.path.join(source_folder, f), out_path):
            success += 1
        else:
            fail += 1
    return success, fail

spec_asv_base = "D:/Deepfake-detection/datasets/spectrograms_asvspoof"

splits_classes = [
    ("train", "bonafide"), ("train", "spoof"),
    ("validation", "bonafide"), ("validation", "spoof"),
]

for split, cls in splits_classes:
    source = f"{asv_base}/{split}/{cls}"
    dest = f"{spec_asv_base}/{split}/{cls}"
    success, fail = convert_folder_v3(source, dest)
    print(f"{split}/{cls}: {success} converted, {fail} failed")

Converting bonafide: 100%|██████████| 2500/2500 [04:29<00:00,  9.26it/s]


train/bonafide: 2500 converted, 0 failed


Converting spoof: 100%|██████████| 2500/2500 [06:32<00:00,  6.38it/s]


train/spoof: 2500 converted, 0 failed


Converting bonafide: 100%|██████████| 2500/2500 [09:32<00:00,  4.36it/s] 


validation/bonafide: 2500 converted, 0 failed


Converting spoof: 100%|██████████| 2500/2500 [13:35<00:00,  3.07it/s]

validation/spoof: 2500 converted, 0 failed


Training started

Step 1 - Data loaders
Note: classes will likely show as ['bonafide', 'spoof'] (alphabetical) — so index 0 = bonafide (real), index 1 = spoof (fake). Keep this in mind, it's the reverse order from your face/FoR models where real was index 1.

In [16]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

IMG_SIZE = 224
BATCH_SIZE = 16

asv_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

asv_train_data = datasets.ImageFolder(f"{spec_asv_base}/train", transform=asv_transform)
asv_val_data = datasets.ImageFolder(f"{spec_asv_base}/validation", transform=asv_transform)

asv_train_loader = DataLoader(asv_train_data, batch_size=BATCH_SIZE, shuffle=True)
asv_val_loader = DataLoader(asv_val_data, batch_size=BATCH_SIZE, shuffle=False)

print("Classes:", asv_train_data.classes)
print("Train:", len(asv_train_data), "Val:", len(asv_val_data))

Using device: cuda
Classes: ['bonafide', 'spoof']
Train: 5000 Val: 5000


Step 2 - Model definition

In [17]:
asv_model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
num_features = asv_model.classifier[1].in_features
asv_model.classifier[1] = nn.Linear(num_features, 2)
asv_model = asv_model.to(device)

asv_criterion = nn.CrossEntropyLoss()
asv_optimizer = torch.optim.Adam(asv_model.parameters(), lr=0.0001)

Step 3 - Training/validation functions

In [18]:
def asv_train_one_epoch():
    asv_model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in asv_train_loader:
        images, labels = images.to(device), labels.to(device)
        asv_optimizer.zero_grad()
        outputs = asv_model(images)
        loss = asv_criterion(outputs, labels)
        loss.backward()
        asv_optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

def asv_validate():
    asv_model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in asv_val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = asv_model(images)
            loss = asv_criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return running_loss / total, correct / total

Step 4 - Training loop

In [19]:
EPOCHS = 15
best_val_acc_asv = 0.0

for epoch in range(EPOCHS):
    start = time.time()
    train_loss, train_acc = asv_train_one_epoch()
    val_loss, val_acc = asv_validate()
    elapsed = time.time() - start
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Time: {elapsed:.1f}s")
    
    if val_acc > best_val_acc_asv:
        best_val_acc_asv = val_acc
        torch.save(asv_model.state_dict(), "D:/Deepfake-detection/saved_models/asvspoof_efficientnet_best.pth")
        print(f"  -> New best ASVspoof model saved (val_acc={val_acc:.4f})")

print("ASVspoof model training complete. Best val accuracy:", best_val_acc_asv)

Epoch 1/15 | Train Acc: 0.9472 | Val Acc: 0.9984 | Time: 328.7s
  -> New best ASVspoof model saved (val_acc=0.9984)
Epoch 2/15 | Train Acc: 0.9962 | Val Acc: 0.9994 | Time: 249.8s
  -> New best ASVspoof model saved (val_acc=0.9994)
Epoch 3/15 | Train Acc: 0.9986 | Val Acc: 0.9992 | Time: 257.4s
Epoch 4/15 | Train Acc: 0.9994 | Val Acc: 1.0000 | Time: 254.8s
  -> New best ASVspoof model saved (val_acc=1.0000)
Epoch 5/15 | Train Acc: 0.9970 | Val Acc: 0.9994 | Time: 254.4s
Epoch 6/15 | Train Acc: 0.9990 | Val Acc: 0.9998 | Time: 244.4s
Epoch 7/15 | Train Acc: 0.9998 | Val Acc: 0.9998 | Time: 238.0s
Epoch 8/15 | Train Acc: 0.9996 | Val Acc: 0.9998 | Time: 221.2s
Epoch 9/15 | Train Acc: 1.0000 | Val Acc: 1.0000 | Time: 227.6s
Epoch 10/15 | Train Acc: 0.9992 | Val Acc: 0.9994 | Time: 242.9s
Epoch 11/15 | Train Acc: 0.9996 | Val Acc: 0.9996 | Time: 222.2s
Epoch 12/15 | Train Acc: 0.9992 | Val Acc: 0.9998 | Time: 218.2s
Epoch 13/15 | Train Acc: 0.9996 | Val Acc: 0.9990 | Time: 216.3s
Epoch 14

 Step 5 - eval — first, extract a balanced sample

the eval set uses different TTS/voice-conversion systems than train/dev — specifically so that a model can't just memorize quirks of the exact synthesis systems it trained on. This is a genuinely harder, more meaningful generalization test than a random held-out split, and it's built into the dataset by the original researchers for exactly this purpose

In [6]:
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    eval_labels_map = parse_protocol(zip_ref, "LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt")

print("Eval labels:", len(eval_labels_map))
from collections import Counter
print("Eval balance:", Counter(eval_labels_map.values()))

eval_bonafide, eval_spoof = get_balanced_filenames(eval_labels_map, samples_per_class=1000)
print(f"Sampled: {len(eval_bonafide)} bonafide, {len(eval_spoof)} spoof")

Eval labels: 71237
Eval balance: Counter({'spoof': 63882, 'bonafide': 7355})
Sampled: 1000 bonafide, 1000 spoof


Step 6 - extract these eval audio files and run the real generalization test

In [7]:
eval_entries = [e for e in la_audio_entries if "ASVspoof2019_LA_eval" in e]
eval_lookup = build_filename_lookup(eval_entries)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    e1, m1 = extract_selected(zip_ref, eval_bonafide, eval_lookup, f"{asv_base}/eval/bonafide")
    e2, m2 = extract_selected(zip_ref, eval_spoof, eval_lookup, f"{asv_base}/eval/spoof")

print(f"eval/bonafide: {e1} extracted, {m1} missing")
print(f"eval/spoof: {e2} extracted, {m2} missing")

Extracting to spoof: 100%|██████████| 1000/1000 [00:02<00:00, 423.08it/s]

eval/bonafide: 1000 extracted, 0 missing
eval/spoof: 1000 extracted, 0 missing


Step 7 - convert to spectrograms and run the actual test

In [8]:
for cls in ["bonafide", "spoof"]:
    source = f"{asv_base}/eval/{cls}"
    dest = f"{spec_asv_base}/eval/{cls}"
    success, fail = convert_folder_v3(source, dest)
    print(f"eval/{cls}: {success} converted, {fail} failed")

asv_eval_data = datasets.ImageFolder(f"{spec_asv_base}/eval", transform=asv_transform)
asv_eval_loader = DataLoader(asv_eval_data, batch_size=BATCH_SIZE, shuffle=False)

def asv_evaluate(loader, model, class_names, name="Eval"):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    print(f"--- {name} ---")
    print(classification_report(all_labels, all_preds, target_names=class_names))
    unique, counts = np.unique(all_preds, return_counts=True)
    print("Prediction distribution:", dict(zip(unique, counts)))

asv_evaluate(asv_eval_loader, asv_model, asv_train_data.classes, name="ASVspoof Eval Set (different TTS systems)")

Converting bonafide: 100%|██████████| 1000/1000 [01:59<00:00,  8.35it/s]


eval/bonafide: 1000 converted, 0 failed


Converting spoof: 100%|██████████| 1000/1000 [01:59<00:00,  8.37it/s]


eval/spoof: 1000 converted, 0 failed
--- ASVspoof Eval Set (different TTS systems) ---
              precision    recall  f1-score   support

    bonafide       0.81      1.00      0.90      1000
       spoof       1.00      0.77      0.87      1000

    accuracy                           0.89      2000
   macro avg       0.91      0.89      0.88      2000
weighted avg       0.91      0.89      0.88      2000

Prediction distribution: {np.int64(0): np.int64(1230), np.int64(1): np.int64(770)}
